# CNU Campus ChatBot — 보조 제출 노트북 (Gemma 3 / torch 2.5.1)

메인(Gemma 4 + torch 2.11)이 막힐 경우를 대비한 **백업 제출본**.
**torch 2.5.1 + transformers 4.50 + Gemma 3 12B(4bit, bfloat16)** 조합으로
RAG 챗봇을 구동하고 `outputs/chat_output.json` 생성 + Gradio UI 실행.

## ⚠️ 사전 준비 (중요)
벡터DB는 HF Hub `adoveflash/cnu-qa-system`에서 받는다. **최신 코퍼스(식단·학사일정 복구본)가
그 repo에 업로드돼 있어야** 한다. (박스에서 `upload_vectordb` 스크립트로 먼저 올릴 것)

## 실행 순서 (Colab T4, 새 세션)
1. **[1] torch 2.5.1 설치** → 자동 재시작
2. **[2] 레포 clone + 라이브러리 설치** → 자동 재시작
3. **[3] HF 로그인 + 벡터DB 다운로드** (gemma-3 gated)
4. **[4] Gemma 3 로드** (bfloat16 필수)
5. **[5] RAG 검색 설정**
6. **[6] 답변 생성 함수**
7. **[7] 배치 추론 → chat_output.json**
8. **[8] Gradio UI**

## [1] torch 2.5.1 설치 → 자동 재시작

In [ ]:
import os
!pip install -q torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1
print("torch 2.5.1 설치 완료 — 재시작합니다. 재시작되면 [2]부터.")
os.kill(os.getpid(), 9)

## [2] (재시작 후) 레포 clone + 라이브러리 설치 → 자동 재시작

In [ ]:
import os
REPO = "cnu_qa_system"
if not os.path.exists(REPO):
    !git clone https://github.com/adoveflash/cnu_qa_system.git
if not os.getcwd().endswith(REPO):
    os.chdir(REPO)
print("cwd:", os.getcwd())

os.environ["HF_HOME"] = "/content/hf_cache"  # Drive 아님 (쿼터 회피)
os.makedirs(os.environ["HF_HOME"], exist_ok=True)

import torch
assert torch.__version__.startswith("2.5.1"), "torch 2.5.1 아님 — [1]부터 다시"

!pip install -q transformers==4.50.0 accelerate "bitsandbytes>=0.45" \
    sentence-transformers chromadb gradio huggingface_hub \
    torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1

# torchcodec은 Colab 기본(torch 2.11)용이라 2.5.1과 비호환 — 텍스트엔 불필요하므로 제거
!pip uninstall -y -q torchcodec

print("\n설치 완료 — 재시작합니다. 재시작되면 [3]부터.")
os.kill(os.getpid(), 9)

## [3] (재시작 후) HF 로그인 + 벡터DB 다운로드\ngemma-3는 gated → 라이선스 수락 + 토큰 필요.

In [ ]:
import os
os.chdir("/content/cnu_qa_system")
os.environ["HF_HOME"] = "/content/hf_cache"

from huggingface_hub import login
login()  # gemma-3 gated — 토큰 입력 (huggingface.co/google/gemma-3-12b-it 라이선스 먼저 수락)

from huggingface_hub import snapshot_download
HF_REPO = "adoveflash/cnu-qa-system"
VECTOR_DB_PATH = "data/vector_db"
if not os.path.exists(VECTOR_DB_PATH):
    snapshot_download(repo_id=HF_REPO, local_dir=".", allow_patterns=["data/vector_db/**"])
print("벡터DB 준비 완료")

## [4] Gemma 3 로드 (bfloat16 필수 — float16이면 NaN)

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # 단편화 완화 (CUDA 초기화 전 설정)

import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

MODEL_ID = "google/gemma-3-12b-it"   # 더 가볍게: "google/gemma-3-4b-it"
bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,   # Gemma는 bfloat16 (float16=NaN/pad)
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, quantization_config=bnb,
    device_map={"": 0},          # auto 대신 통째로 GPU0 (CPU offload 방지)
    torch_dtype=torch.bfloat16,
)
model.eval()
print(f"모델 로드 완료 — VRAM {torch.cuda.memory_reserved()/1e9:.2f} GB | torch {torch.__version__}")

## [5] RAG 검색 (bge-m3 CPU + ChromaDB)

In [ ]:
from sentence_transformers import SentenceTransformer
import chromadb

print("임베딩 모델 로드: BAAI/bge-m3 (CPU)")
embed_model = SentenceTransformer("BAAI/bge-m3", device="cpu")
client = chromadb.PersistentClient(path="data/vector_db")
collection = client.get_collection("cnu_chunks")
print(f"벡터DB: {collection.count()}개 청크")

def build_context(query, top_k=5):
    emb = embed_model.encode([query]).tolist()[0]
    res = collection.query(query_embeddings=[emb], n_results=top_k,
                           include=["documents", "metadatas"])
    parts, urls = [], []
    for i in range(len(res["ids"][0])):
        meta = res["metadatas"][0][i]
        parts.append(f"[참고{i+1}] {meta.get('title','')}\n{res['documents'][0][i]}")
        u = meta.get("url", "")
        if u and u not in urls:
            urls.append(u)
    return "\n\n".join(parts), urls

## [6] 답변 생성 (system을 user에 합침 — Gemma 3는 system role 미지원)

In [ ]:
import torch, re, gc

SYSTEM = (
    "너는 충남대학교 학내 정보를 안내하는 친절한 AI 챗봇이야. "
    "참고 자료에 있는 내용(날짜·학점·일정 등 구체 수치 포함)만으로 답하고, 없으면 "
    "'확인되지 않았어요'라고 솔직히 답해. 답변에 '[참고1]' 같은 내부 표시를 쓰지 말고 "
    "자연스러운 한국어로만 답해."
)

def generate_answer(question, context, urls, max_new_tokens=384):
    user = (f"{SYSTEM}\n\n참고 자료:\n{context}\n\n질문: {question}"
            if context else f"{SYSTEM}\n\n질문: {question}")
    msgs = [{"role": "user", "content": [{"type": "text", "text": user}]}]
    inputs = processor.apply_chat_template(
        msgs, add_generation_prompt=True, tokenize=True,
        return_dict=True, return_tensors="pt").to(model.device)
    n = inputs["input_ids"].shape[-1]
    try:
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                                 do_sample=False, repetition_penalty=1.3)
        ans = processor.decode(out[0][n:], skip_special_tokens=True).strip()
        ans = re.sub(r"\[\s*참고\s*\d+\s*\]", "", ans).strip()
    finally:
        del inputs
        gc.collect(); torch.cuda.empty_cache()   # 매 호출 후 캐시 정리 (단편화/OOM 방지)
    return ans

# 테스트
ctx, urls = build_context("컴퓨터융합학부 졸업하려면 몇 학점이야?")
print(generate_answer("컴퓨터융합학부 졸업하려면 몇 학점이야?", ctx, urls))

## [7] 배치 추론 → outputs/chat_output.json

In [ ]:
import json, os, time
os.makedirs("outputs", exist_ok=True)
TEST = "data/test_chat.json"
if os.path.exists(TEST):
    with open(TEST, encoding="utf-8") as f:
        test_data = json.load(f)
    results = []
    for i, item in enumerate(test_data):
        q = item["user"]
        t = time.time()
        ctx, urls = build_context(q)
        ans = generate_answer(q, ctx, urls)
        results.append({"id": item.get("id", i), "user": q, "model": ans})
        print(f"[{i+1}/{len(test_data)}] {q[:30]}... ({time.time()-t:.0f}s, {len(ans)}자)")
    with open("outputs/chat_output.json", "w", encoding="utf-8") as f:
        json.dump(results, f, ensure_ascii=False, indent=2)
    print("\n저장: outputs/chat_output.json")
else:
    print(f"{TEST} 없음 — 평가 시 조교가 제공. (배치 건너뜀)")

## [8] Gradio UI

In [ ]:
import gradio as gr

def chat_fn(message, history):
    ctx, urls = build_context(message)
    ans = generate_answer(message, ctx, urls)
    if urls:
        ans += "\n\n참고: " + ", ".join(urls[:3])
    return ans

gr.ChatInterface(
    chat_fn,
    title="CNU Campus AI (Gemma 3 / torch 2.5.1 백업)",
    description="충남대학교 학내 정보 Q&A 챗봇",
    examples=["컴퓨터융합학부 졸업 요건", "오늘 학생회관 점심 메뉴", "셔틀버스 시간표"],
).launch(share=True)